# A2 — Pesquisa Operacional I · Caminho Mínimo na rede viária de Kraków (Dijkstra e A*)

**Trilha:** 3 — Caminho Mínimo em Grafo (Dijkstra/A*) · **Dados:** rede viária real de Kraków (OpenStreetMap, via OSMnx).  
Este notebook **amplia o trabalho da rede Polska (12 nós)** para uma rede real com dezenas de milhares de cruzamentos. A Polska continua no repositório como validação de corretude.

### Como executar (Google Colab)
1. Suba este `.ipynb` no Colab e use **Ambiente de execução → Executar tudo**. A 1ª execução baixa Kraków do OpenStreetMap (1 a 3 min) e salva `krakow_drive.graphml`; as próximas usam o arquivo salvo.
2. **Caso real (opcional):** se subir um `pontos_empresa.csv` (colunas `nome,lat,lon`) com pontos reais de uma empresa (entregas, filiais, clientes), todo o cenário "Antes × Depois" passa a usar esses pontos. Sem o arquivo, usa pares origem–destino **amostrados** — e o notebook avisa que isso **não** é caso real.

### Roteiro
| Seção | Conteúdo | Critério |
|---|---|---|
| 1 | Dados do OSM e **validação automática** das restrições | 1.1 / 2.2 |
| 2 | Dijkstra e A* do zero + **verificação cruzada** com NetworkX | 2.2 |
| 3 | **Passo a passo manual** com um trecho real da cidade | 2.1 |
| 4 | **Análise de tempo** e escalabilidade (O(V²) × heap × A*) | 2.2 |
| 5 | **Painel executivo Antes × Depois** | 2.3 / 2.5 |
| 6 | **Sensibilidade**: via bloqueada e erro nos pesos | 2.5 |
| 7 | Alternativas algorítmicas e resumo executivo | 2.1 / 2.5 |

## 0. Modelagem formal
Grafo **direcionado** e ponderado $G=(V,A,w)$ (ruas de mão única existem):

* **Vértices** $V$: cruzamentos/pontas de rua. **Arcos** $A$: trechos de via dirigíveis $(u\to v)$. **Peso** $w_{uv}>0$: comprimento do trecho em **metros** (atributo `length` do OSM).
* **Variável de decisão:** $x_{uv}\in\{0,1\}$ — 1 se o trecho $u\to v$ faz parte da rota.
* **Função objetivo:** $\min \sum_{(u,v)\in A} w_{uv}\,x_{uv}$
* **Restrições:** conservação de fluxo — saída líquida $=1$ na origem $s$, $=-1$ no destino $t$, $=0$ nos demais nós; $x\in\{0,1\}$.
* **Por que Dijkstra:** pesos estritamente positivos (validado na seção 1) → escolha gulosa é ótima; $O((V+A)\log V)$ com fila de prioridade.
* **Por que também A\*:** os pesos são distâncias reais, então a distância em linha reta (haversine) é uma **heurística admissível** — resolve o mesmo problema expandindo menos nós.

In [ ]:
import json, math, heapq, time, random, os, subprocess, sys, statistics
from pathlib import Path
from collections import deque
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
import networkx as nx
try:
    import osmnx as ox
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "osmnx"], check=True)
    import osmnx as ox

# ----------------------------- PARÂMETROS -----------------------------
CIDADE        = "Kraków, Poland"
ARQ_CACHE     = Path("krakow_drive.graphml")      # cache local do grafo baixado
ARQ_PONTOS    = Path("pontos_empresa.csv")        # CASO REAL (opcional): nome,lat,lon
SEMENTE       = 2026
N_PARES       = 150                               # pares origem–destino do cenário (amostra)
VEL_MEDIA_KMH = 30.0                              # HIPÓTESE (tráfego urbano) — ajuste se tiver dado real
CUSTO_POR_KM  = None                              # ex.: 1.80 (R$/km) se a empresa informar; None = não calcula
random.seed(SEMENTE); np.random.seed(SEMENTE)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 1. Dados (OpenStreetMap) e validação das restrições

**Origem:** OpenStreetMap, rede `drive` de Kraków obtida com `osmnx.graph_from_place`. **Colunas usadas:** nós `x` (longitude) e `y` (latitude); arcos `length` (m) e a orientação (mão única). **Por que esta base:** é pública, real, tem a escala de uma operação logística urbana e os pesos têm significado físico (metros), o que permite validar a solução e usar A\*.

In [ ]:
if ARQ_CACHE.is_file():
    G_osm = ox.load_graphml(ARQ_CACHE)
else:
    G_osm = ox.graph_from_place(CIDADE, network_type="drive")
    ox.save_graphml(G_osm, ARQ_CACHE)
print(f"Bruto do OSM: {G_osm.number_of_nodes():,} nós · {G_osm.number_of_edges():,} arcos")

def montar_grafo(G):
    # 1) maior componente fortemente conexa (toda origem alcança todo destino)
    comp = max(nx.strongly_connected_components(G), key=len)
    H = G.subgraph(comp)
    ids = {n: i for i, n in enumerate(H.nodes)}
    melhor = {}
    for u, v, d in H.edges(data=True):          # 2) arcos paralelos -> mantém o menor; remove laços e pesos <= 0
        w = float(d.get("length", 0.0))
        if u == v or w <= 0: continue
        k = (ids[u], ids[v])
        if k not in melhor or w < melhor[k]: melhor[k] = w
    grafo = {i: [] for i in ids.values()}
    for (a, b), w in melhor.items(): grafo[a].append((b, w))
    coords = {ids[n]: (float(H.nodes[n]["x"]), float(H.nodes[n]["y"])) for n in H.nodes}   # (lon, lat)
    return grafo, coords

grafo, coords = montar_grafo(G_osm)
V = len(grafo); A = sum(len(x) for x in grafo.values())
print(f"Rede usada: {V:,} nós · {A:,} arcos")

In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def alcanca_todos(g, inicio, reverso=False):
    adj = g
    if reverso:
        adj = {n: [] for n in g}
        for u in g:
            for v, w in g[u]: adj[v].append((u, w))
    vistos = {inicio}; fila = deque([inicio])
    while fila:
        u = fila.popleft()
        for v, _ in adj[u]:
            if v not in vistos: vistos.add(v); fila.append(v)
    return len(vistos) == len(g)

def validar(grafo, coords):
    arcos = [(u, v, w) for u in grafo for v, w in grafo[u]]
    conj = {(u, v) for u, v, _ in arcos}
    viol = sum(1 for u, v, w in arcos if w < 0.99 * haversine_m(*coords[u], *coords[v]))
    mao_unica = sum(1 for u, v, _ in arcos if (v, u) not in conj) / len(arcos)
    testes = [
        ("Pesos estritamente positivos (requisito do Dijkstra)", all(w > 0 for *_, w in arcos)),
        ("Sem laços (u != v)",                                    all(u != v for u, v, _ in arcos)),
        ("Sem arcos duplicados",                                   len(conj) == len(arcos)),
        ("Extremos de todo arco existem",                          all(v in grafo for _, v, _ in arcos)),
        ("Todo nó tem coordenadas",                                set(coords) == set(grafo)),
        ("Fortemente conexo (ida: todos alcançáveis a partir do nó 0)", alcanca_todos(grafo, next(iter(grafo)))),
        ("Fortemente conexo (volta: todos alcançam o nó 0)",       alcanca_todos(grafo, next(iter(grafo)), reverso=True)),
        ("Comprimento do trecho ≥ distância em linha reta (±1%) em ≥ 99% dos arcos", viol / len(arcos) < 0.01),
    ]
    df = pd.DataFrame(testes, columns=["Restrição do modelo", "Atendida"])
    assert df["Atendida"].all(), "Base viola o modelo:\n" + df.to_string()
    print(f"Arcos de mão única: {100*mao_unica:.1f}% · comprimento médio do trecho: {np.mean([w for *_, w in arcos]):.1f} m")
    return df
validar(grafo, coords)

## 2. Algoritmos (do zero) e verificação cruzada

In [ ]:
def dijkstra(grafo, origem, destino=None, proibidos=frozenset(), pesos=None):
    # Dijkstra com fila de prioridade: O((V+A) log V). Para ao fechar o destino. Retorna (dist, ant, nós_expandidos)
    dist = {origem: 0.0}; ant = {origem: None}; fila = [(0.0, origem)]; fechados = set(); exp = 0
    while fila:
        d, u = heapq.heappop(fila)
        if u in fechados: continue
        fechados.add(u); exp += 1
        if u == destino: break
        for v, w in grafo[u]:
            if proibidos and (u, v) in proibidos: continue
            nd = d + w
            if nd < dist.get(v, math.inf): dist[v] = nd; ant[v] = u; heapq.heappush(fila, (nd, v))
    return dist, ant, exp

def dijkstra_n2(grafo, origem, destino=None):
    # Versão didática do Main.py: seleção linear do menor, O(V²)
    dist = {n: math.inf for n in grafo}; ant = {n: None for n in grafo}; abertos = set(grafo); dist[origem] = 0.0
    while abertos:
        atual = min(abertos, key=lambda n: dist[n])
        if dist[atual] == math.inf: break
        abertos.remove(atual)
        if atual == destino: break
        for viz, w in grafo[atual]:
            if viz in abertos and dist[atual] + w < dist[viz]: dist[viz] = dist[atual] + w; ant[viz] = atual
    return dist, ant

def astar(grafo, coords, origem, destino, fator):
    tx, ty = coords[destino]
    h = lambda n: fator * haversine_m(coords[n][0], coords[n][1], tx, ty)
    g = {origem: 0.0}; ant = {origem: None}; fila = [(h(origem), origem)]; fechados = set(); exp = 0
    while fila:
        _, u = heapq.heappop(fila)
        if u in fechados: continue
        fechados.add(u); exp += 1
        if u == destino: break
        for v, w in grafo[u]:
            ng = g[u] + w
            if ng < g.get(v, math.inf): g[v] = ng; ant[v] = u; heapq.heappush(fila, (ng + h(v), v))
    return g.get(destino, math.inf), ant, exp

def reconstruir(ant, origem, destino):
    caminho, x = [], destino
    while x is not None:
        if x not in ant: return None
        caminho.append(x); x = ant[x]
    caminho.reverse()
    return caminho if caminho[0] == origem else None

PESO = {(u, v): w for u in grafo for v, w in grafo[u]}
def comprimento(caminho): return sum(PESO[(a, b)] for a, b in zip(caminho, caminho[1:]))

# fator de segurança da heurística: garante h(u) <= w(u,v) + h(v) em TODOS os arcos (consistência)
razoes = [w / max(haversine_m(*coords[u], *coords[v]), 1e-9) for (u, v), w in PESO.items() if haversine_m(*coords[u], *coords[v]) > 1.0]
FATOR_H = min(1.0, min(razoes))
print(f"Heurística A*: fator de segurança = {FATOR_H:.3f} (1,000 = linha reta pura)")

In [ ]:
# Verificação cruzada em 200 pares aleatórios: Dijkstra × A* × NetworkX (oráculo) + validade do caminho
G_nx = nx.DiGraph(); G_nx.add_weighted_edges_from([(u, v, w) for (u, v), w in PESO.items()])
nos = list(grafo); pares_teste = [tuple(random.sample(nos, 2)) for _ in range(200)]
for s, t in pares_teste:
    dist, ant, _ = dijkstra(grafo, s, t)
    d_a, ant_a, _ = astar(grafo, coords, s, t, FATOR_H)
    ref = nx.dijkstra_path_length(G_nx, s, t, weight="weight")
    assert math.isclose(dist[t], ref, rel_tol=1e-9, abs_tol=1e-6), (s, t, dist[t], ref)
    assert math.isclose(d_a, ref, rel_tol=1e-9, abs_tol=1e-6), (s, t, d_a, ref)
    c = reconstruir(ant, s, t)
    assert c[0] == s and c[-1] == t and len(set(c)) == len(c) and math.isclose(comprimento(c), ref, rel_tol=1e-9)
print("200 pares: Dijkstra = A* = NetworkX; caminhos simples, contínuos e com custo = soma dos pesos. 0 divergências.")

## 3. Como o Dijkstra calcula — passo a passo manual com um trecho real da cidade
A cidade inteira tem dezenas de milhares de nós, então a demonstração manual usa um **recorte real de 12 cruzamentos** perto da Rynek Główny (centro histórico), com os mesmos comprimentos em metros e as mesmas mãos únicas. O código é o mesmo da seção 2.

In [ ]:
def mais_proximo(lat, lon):
    ids = np.fromiter(coords.keys(), dtype=np.int64)
    xy = np.array([coords[i] for i in ids]); k = math.cos(math.radians(lat))
    d = (xy[:, 0] - lon) ** 2 * k * k + (xy[:, 1] - lat) ** 2
    return int(ids[int(np.argmin(d))])

def recorte_bola(grafo, centro, n):
    ordem = [centro]; vistos = {centro}; fila = deque([centro])
    while fila and len(ordem) < n:
        u = fila.popleft()
        for v, _ in grafo[u]:
            if v not in vistos and len(ordem) < n: vistos.add(v); ordem.append(v); fila.append(v)
    sub = {u: [(v, w) for v, w in grafo[u] if v in vistos] for u in ordem}
    return sub, ordem

CENTRO = mais_proximo(50.0617, 19.9373)          # Rynek Główny
sub, ordem = recorte_bola(grafo, CENTRO, 12)
rot = {n: f"N{i+1}" for i, n in enumerate(ordem)}  # rótulos curtos
O_EX, D_EX = ordem[0], ordem[-1]

def dijkstra_trace(g, origem, destino):
    dist = {n: math.inf for n in g}; ant = {n: None for n in g}; abertos = set(g); fechados = []; dist[origem] = 0.0; passos = []
    while abertos:
        atual = min(abertos, key=lambda n: dist[n])
        if dist[atual] == math.inf: break
        abertos.remove(atual); fechados.append(atual); relax = []
        if atual != destino:
            for viz, w in g[atual]:
                if viz in abertos:
                    nova = dist[atual] + w; ok = nova < dist[viz]
                    relax.append((viz, dist[atual], w, nova, dist[viz], ok))
                    if ok: dist[viz] = nova; ant[viz] = atual
        passos.append(dict(atual=atual, dist=dict(dist), fechados=list(fechados), relax=relax, ant=dict(ant)))
        if atual == destino: break
    return passos

passos = dijkstra_trace(sub, O_EX, D_EX)
print(f"Recorte real: {len(sub)} nós, {sum(len(x) for x in sub.values())} arcos · origem {rot[O_EX]} → destino {rot[D_EX]}\n")
for i, p in enumerate(passos, 1):
    print(f"Iteração {i}: fecha {rot[p['atual']]} (menor custo conhecido = {p['dist'][p['atual']]:.1f} m)")
    for viz, base, w, nova, antiga, ok in p["relax"]:
        a = "∞" if antiga == math.inf else f"{antiga:.1f}"
        print(f"     {rot[viz]:<4}: {base:8.1f} + {w:7.1f} = {nova:8.1f}  vs {a:>8}  → {'ATUALIZA ✔' if ok else 'mantém'}")

In [ ]:
linhas = []
for i, p in enumerate(passos, 1):
    l = {"Iter.": i, "Fecha": rot[p["atual"]]}
    for n in ordem:
        v = p["dist"][n]; l[rot[n]] = ("∞" if v == math.inf else f"{v:.0f}") + ("*" if n in p["fechados"] else "")
    linhas.append(l)
display(pd.DataFrame(linhas).set_index("Iter."))

def desenhar_passos(passos, sub, ordem, destino):
    n = len(passos); cols = 4; lin = math.ceil(n / cols)
    fig, axs = plt.subplots(lin, cols, figsize=(4.6 * cols, 3.8 * lin)); axs = np.atleast_1d(axs).ravel()
    k = math.cos(math.radians(50.06)); pos = {n_: (coords[n_][0] * k, coords[n_][1]) for n_ in ordem}
    R = nx.DiGraph(); R.add_nodes_from(ordem)
    for u in sub:
        for v, w in sub[u]: R.add_edge(u, v)
    for ax, p, i in zip(axs, passos, range(1, n + 1)):
        cor = ["#e32636" if x == p["atual"] else "#20bf55" if x in p["fechados"] else "#ffd43b" if p["dist"][x] < math.inf else "#d0d7de" for x in R.nodes]
        nx.draw_networkx_edges(R, pos, ax=ax, edge_color="#8aa0ae", width=1.0, arrows=True, arrowsize=9, node_size=380)
        arv = [(a, b) for b, a in p["ant"].items() if a is not None]
        nx.draw_networkx_edges(R, pos, edgelist=arv, ax=ax, edge_color="#17324d", width=2.4, arrows=True, arrowsize=11, node_size=380)
        nx.draw_networkx_nodes(R, pos, ax=ax, node_color=cor, node_size=380, edgecolors="#183247")
        rr = {x: (f"{rot[x]}\n{p['dist'][x]:.0f}" if p["dist"][x] < math.inf else f"{rot[x]}\n∞") for x in R.nodes}
        nx.draw_networkx_labels(R, pos, labels=rr, ax=ax, font_size=6, font_weight="bold")
        ax.set_title(f"Iter. {i}: fecha {rot[p['atual']]}", fontsize=9); ax.axis("off")
    for ax in axs[n:]: ax.axis("off")
    fig.suptitle("Dijkstra passo a passo (recorte real) — rótulo = nó e menor distância conhecida em metros; setas = sentido da via", fontsize=11)
    plt.tight_layout(); plt.show()
desenhar_passos(passos, sub, ordem, D_EX)

### Onde cada passo aparece no código (`dijkstra` / `dijkstra_n2`)
| Conceito | Código | O que acontece |
|---|---|---|
| Inicialização | `dist = {origem: 0.0}` (no n2: todos `inf`) | origem custa 0; o resto é desconhecido |
| Escolha gulosa | `heappop` (n2: `min(abertos, key=...)`) | fecha o nó de menor custo conhecido; esse custo não muda mais (pesos > 0) |
| Relaxamento | `nd = d + w; if nd < dist.get(v, inf)` | compara o caminho novo com o melhor conhecido — as contas impressas acima |
| Predecessor | `ant[v] = u` | guarda de onde veio o melhor custo — setas escuras nos mapas |
| Parada antecipada | `if u == destino: break` | em redes grandes isso evita explorar a cidade toda |
| Reconstrução | `reconstruir(ant, origem, destino)` | segue `ant` do destino à origem e inverte |

## 4. Análise de tempo e escalabilidade

In [ ]:
def mediana_s(f, rep=5):
    ts = []
    for _ in range(rep):
        t0 = time.perf_counter(); f(); ts.append(time.perf_counter() - t0)
    return statistics.median(ts)

pares_t = [tuple(random.sample(nos, 2)) for _ in range(60)]
t_full = mediana_s(lambda: dijkstra(grafo, pares_t[0][0]), 3)
t_dij = statistics.mean(mediana_s(lambda s=s, t=t: dijkstra(grafo, s, t), 1) for s, t in pares_t)
t_ast = statistics.mean(mediana_s(lambda s=s, t=t: astar(grafo, coords, s, t, FATOR_H), 1) for s, t in pares_t)
t_nx  = statistics.mean(mediana_s(lambda s=s, t=t: nx.dijkstra_path_length(G_nx, s, t), 1) for s, t in pares_t[:20])
exp_d = [dijkstra(grafo, s, t)[2] for s, t in pares_t]; exp_a = [astar(grafo, coords, s, t, FATOR_H)[2] for s, t in pares_t]
print(f"1 origem → TODOS os {V:,} nós (heap): {t_full*1e3:.0f} ms")
print(f"Par origem–destino (média de {len(pares_t)} pares):  Dijkstra {t_dij*1e3:.1f} ms · A* {t_ast*1e3:.1f} ms · NetworkX {t_nx*1e3:.1f} ms")
print(f"Nós expandidos (média): Dijkstra {np.mean(exp_d):,.0f} · A* {np.mean(exp_a):,.0f}  →  A* expande {100*(1-np.mean(exp_a)/np.mean(exp_d)):.0f}% menos nós")

In [ ]:
# Escalabilidade O(V²) × heap em recortes reais crescentes da própria cidade (1 origem → todos)
res = []
for n in [250, 500, 1000, 2000]:
    sg, _ = recorte_bola(grafo, CENTRO, n); Vn = len(sg); An = sum(len(x) for x in sg.values()); o = next(iter(sg))
    tn = mediana_s(lambda: dijkstra_n2(sg, o), 3); th = mediana_s(lambda: dijkstra(sg, o), 3)
    res.append((Vn, An, tn * 1e3, th * 1e3))
res.append((V, A, np.nan, t_full * 1e3))
df_t = pd.DataFrame(res, columns=["V", "A", "O(V²) ms", "heap ms"]).round(2); display(df_t)
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.loglog(df_t["V"], df_t["O(V²) ms"], "o-", label="Dijkstra O(V²) (versão do Main.py)")
ax.loglog(df_t["V"], df_t["heap ms"], "s-", label="Dijkstra com heap")
ax.set_xlabel("Nós (V)"); ax.set_ylabel("Tempo, 1 origem → todos (ms)"); ax.grid(True, which="both", alpha=.3)
ax.set_title("Escalabilidade em recortes reais de Kraków (log-log)"); ax.legend(); plt.tight_layout(); plt.show()

**Leitura:** a versão de lista cresce ~quadraticamente e se torna inviável na cidade inteira (por isso não foi executada em todos os nós); a de heap cresce quase linearmente. O A\* usa a distância em linha reta para expandir bem menos nós **sem perder a otimalidade** (verificado contra o NetworkX na seção 2).

## 5. Antes × Depois
**"Depois" (otimizado):** rota de **menor distância** (Dijkstra).  
**"Antes" (referência sem otimização por distância):** rota de **menor número de trechos** (menos cruzamentos), critério simples que um roteador ingênuo poderia usar. *O tamanho do ganho depende dessa definição de "antes" — por isso ela está explícita.*  
**Pares O–D:** se existir `pontos_empresa.csv`, são os pontos reais da empresa (**caso real**); senão, pares amostrados da própria cidade (**simulação, não é caso real**).

In [ ]:
if ARQ_PONTOS.is_file():
    pts = pd.read_csv(ARQ_PONTOS); assert {"nome", "lat", "lon"} <= set(pts.columns)
    pts["no"] = [mais_proximo(la, lo) for la, lo in zip(pts.lat, pts.lon)]
    todos = [(a, b) for a in pts.index for b in pts.index if a != b]
    random.shuffle(todos); sel = todos[:N_PARES]
    pares = [(pts.no[a], pts.no[b], f"{pts.nome[a]} → {pts.nome[b]}") for a, b in sel]
    CASO_REAL = True
else:
    pares = []
    while len(pares) < N_PARES:
        s, t = random.sample(nos, 2)
        if haversine_m(*coords[s], *coords[t]) > 2000: pares.append((s, t, f"{s} → {t}"))   # viagens com ≥ 2 km em linha reta
    CASO_REAL = False
print("CASO REAL (pontos da empresa)" if CASO_REAL else "SIMULAÇÃO com pares amostrados (NÃO é caso real)", f"— {len(pares)} pares")

def bfs_trechos(grafo, origem):
    ant = {origem: None}; fila = deque([origem])
    while fila:
        u = fila.popleft()
        for v, _ in grafo[u]:
            if v not in ant: ant[v] = u; fila.append(v)
    return ant

cache_bfs = {}; linhas = []
for s, t, nome in pares:
    if s not in cache_bfs: cache_bfs[s] = bfs_trechos(grafo, s)
    c_a = reconstruir(cache_bfs[s], s, t)
    _, ant, _ = dijkstra(grafo, s, t); c_d = reconstruir(ant, s, t)
    linhas.append(dict(par=nome, s=s, t=t, m_antes=comprimento(c_a), m_depois=comprimento(c_d),
                       trechos_antes=len(c_a) - 1, trechos_depois=len(c_d) - 1, c_antes=c_a, c_depois=c_d))
df = pd.DataFrame(linhas)
df["ganho_m"] = df.m_antes - df.m_depois; df["ganho_pct"] = 100 * df.ganho_m / df.m_antes
assert (df.ganho_m > -1e-6).all(), "Dijkstra não pode ser pior que outra rota válida"

KM_A, KM_D = df.m_antes.sum() / 1000, df.m_depois.sum() / 1000
GANHO_PCT = 100 * (KM_A - KM_D) / KM_A; MUDARAM = int((df.ganho_m > 1e-6).sum())
MIN_A, MIN_D = 60 * KM_A / VEL_MEDIA_KMH, 60 * KM_D / VEL_MEDIA_KMH
print(f"Distância total: {KM_A:,.1f} km → {KM_D:,.1f} km · economia {KM_A-KM_D:,.1f} km ({GANHO_PCT:.1f}%)")
print(f"Rotas alteradas: {MUDARAM} de {len(df)} · maior ganho: {df.ganho_pct.max():.1f}% ({df.ganho_m.max():,.0f} m)")
print(f"Tempo a {VEL_MEDIA_KMH:.0f} km/h (HIPÓTESE): {MIN_A:,.0f} → {MIN_D:,.0f} min ({MIN_A-MIN_D:,.0f} min a menos)")
if CUSTO_POR_KM: print(f"Custo a R$ {CUSTO_POR_KM:.2f}/km: R$ {KM_A*CUSTO_POR_KM:,.2f} → R$ {KM_D*CUSTO_POR_KM:,.2f} (economia R$ {(KM_A-KM_D)*CUSTO_POR_KM:,.2f})")
df.sort_values("ganho_m", ascending=False)[["par", "m_antes", "m_depois", "ganho_m", "ganho_pct", "trechos_antes", "trechos_depois"]].head(8).round(1)

In [ ]:
K = math.cos(math.radians(50.06))
def seg(c): return [(coords[a][0] * K, coords[a][1]) for a in c]
fundo = LineCollection([[(coords[u][0] * K, coords[u][1]), (coords[v][0] * K, coords[v][1])] for u in grafo for v, _ in grafo[u]],
                       colors="#c9d3da", linewidths=.35)
mel = df.loc[df.ganho_m.idxmax()]
fig = plt.figure(figsize=(17, 9.5), facecolor="#f4f7f8")
gs = fig.add_gridspec(2, 3, width_ratios=[1.55, 1, 1], height_ratios=[1, 1.15], hspace=.34, wspace=.3)
ax_m = fig.add_subplot(gs[:, 0]); ax_k = fig.add_subplot(gs[0, 1]); ax_h = fig.add_subplot(gs[0, 2]); ax_r = fig.add_subplot(gs[1, 1:])

ax_m.add_collection(fundo)
ax_m.plot(*zip(*seg(mel.c_antes)), color="#1f77b4", lw=3.2, ls="--", label=f"ANTES (menos trechos): {mel.m_antes/1000:.2f} km")
ax_m.plot(*zip(*seg(mel.c_depois)), color="#e32636", lw=2.6, label=f"DEPOIS (Dijkstra): {mel.m_depois/1000:.2f} km")
ax_m.scatter(*zip(*[seg([mel.s])[0], seg([mel.t])[0]]), s=90, c="#17324d", zorder=5)
ax_m.autoscale(); ax_m.set_aspect("equal"); ax_m.axis("off"); ax_m.legend(loc="lower left")
ax_m.set_title(f"Par de maior impacto: −{mel.ganho_m/1000:.2f} km ({mel.ganho_pct:.0f}%)", fontsize=12, fontweight="bold")

ax_k.bar(["Antes", "Depois"], [KM_A, KM_D], color=["#1f77b4", "#e32636"])
for i, v in enumerate([KM_A, KM_D]): ax_k.text(i, v, f"{v:,.0f} km", ha="center", va="bottom", fontweight="bold")
ax_k.set_ylim(0, max(KM_A, KM_D) * 1.12); ax_k.spines[["top", "right"]].set_visible(False)
ax_k.set_title(f"Distância total ({len(df)} viagens)\n−{GANHO_PCT:.1f}%  ({KM_A-KM_D:,.0f} km)", fontsize=11, fontweight="bold")

ax_h.hist(df.ganho_pct, bins=20, color="#e32636", alpha=.85)
ax_h.axvline(df.ganho_pct.median(), color="#17324d", ls="--"); ax_h.spines[["top", "right"]].set_visible(False)
ax_h.set_title(f"Economia por viagem (%)\nmediana {df.ganho_pct.median():.1f}% · {MUDARAM}/{len(df)} rotas mudaram", fontsize=11, fontweight="bold")
ax_h.set_xlabel("% de distância evitada"); ax_h.set_ylabel("nº de viagens")

top = df.sort_values("ganho_m", ascending=False).head(8).iloc[::-1]
ax_r.barh(top.par.str.slice(0, 26), top.m_antes / 1000, color="#1f77b4", alpha=.35, label="Antes (km)")
ax_r.barh(top.par.str.slice(0, 26), top.m_depois / 1000, color="#e32636", label="Depois (km)")
for y, (a, b) in enumerate(zip(top.m_antes, top.m_depois)): ax_r.text(a / 1000 + .1, y, f"−{(a-b)/1000:.2f} km", va="center", fontsize=9, fontweight="bold")
ax_r.set_title("8 viagens com maior economia", fontsize=11, fontweight="bold"); ax_r.legend(loc="lower right")
ax_r.spines[["top", "right"]].set_visible(False); ax_r.set_xlim(0, (top.m_antes.max() / 1000) * 1.2)
fig.suptitle(f"PAINEL EXECUTIVO — Kraków: menor nº de trechos × menor distância (Dijkstra) · {'CASO REAL' if CASO_REAL else 'simulação com pares amostrados'}",
             fontsize=15, fontweight="bold", color="#17324d")
fig.savefig("painel_executivo_krakow.png", dpi=200, bbox_inches="tight"); plt.show()

## 6. Análise de sensibilidade
**(a)** E se uma via da rota for bloqueada (obra, acidente)? **(b)** E se os comprimentos/custos estiverem errados?

In [ ]:
# (a) Bloqueio de cada trecho da rota ótima do par de maior impacto: quanto é o desvio?
base_d = mel.m_depois; trechos = list(zip(mel.c_depois, mel.c_depois[1:])); linhas = []
for (u, v) in trechos:
    dist, ant, _ = dijkstra(grafo, mel.s, mel.t, proibidos=frozenset({(u, v)}))
    d = dist.get(mel.t, math.inf)
    linhas.append(dict(trecho=f"{u}→{v}", comprimento_m=PESO[(u, v)], desvio_m=d - base_d, desvio_pct=100 * (d - base_d) / base_d))
df_b = pd.DataFrame(linhas).sort_values("desvio_m", ascending=False)
print(f"Rota ótima: {len(trechos)} trechos, {base_d/1000:.2f} km. Se UM trecho for bloqueado, o acréscimo é:")
print(f"  mediana {df_b.desvio_m.median():,.0f} m · máximo {df_b.desvio_m.max():,.0f} m ({df_b.desvio_pct.max():.1f}%) · {int((df_b.desvio_m > 1e-6).sum())} de {len(df_b)} bloqueios mudam a rota")
display(df_b.head(6).round(1))
fig, ax = plt.subplots(figsize=(8, 3.6)); ax.bar(range(len(df_b)), df_b.sort_index().desvio_m, color="#e32636")
ax.set_xlabel("Trechos da rota (na ordem do percurso)"); ax.set_ylabel("Desvio se bloqueado (m)")
ax.set_title("Quais trechos da rota são mais críticos?"); ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

In [ ]:
# (b) Erro nos pesos: cada comprimento × U(1-σ, 1+σ). Avalia o custo REAL (pesos originais) da rota escolhida com pesos errados.
SIGMAS, RUNS, NPARES = [0.10, 0.20, 0.30], 15, 25
rng = random.Random(SEMENTE); amostra = [(s, t) for s, t, _ in pares[:NPARES]]
otimo = {(s, t): dijkstra(grafo, s, t)[0][t] for s, t in amostra}
rotas_base = {(s, t): tuple(reconstruir(dijkstra(grafo, s, t)[1], s, t)) for s, t in amostra}
res = []
for sig in SIGMAS:
    mudou, perda = [], []
    for _ in range(RUNS):
        ruidoso = {u: [(v, w * rng.uniform(1 - sig, 1 + sig)) for v, w in grafo[u]] for u in grafo}
        for s, t in amostra:
            c = tuple(reconstruir(dijkstra(ruidoso, s, t)[1], s, t))
            mudou.append(c != rotas_base[(s, t)]); perda.append(100 * (comprimento(c) - otimo[(s, t)]) / otimo[(s, t)])
    res.append((f"±{int(sig*100)}%", 100 * np.mean(mudou), np.mean(perda), np.max(perda)))
df_r = pd.DataFrame(res, columns=["Erro nos pesos", "% de viagens com rota diferente", "Distância extra média (%)", "Pior caso (%)"]).round(2)
display(df_r)

**Leitura:** a tabela (a) mostra onde a rota depende de poucos trechos (se um bloqueio causa desvio grande, ali falta alternativa). A tabela (b) mostra o quanto erros de medição de comprimento/custo fazem o roteador escolher rotas piores — e quão pior elas ficam de verdade. Mesmo quando a rota muda, o custo real extra costuma ser pequeno: a solução é **robusta**.

## 7. Alternativas algorítmicas e resumo executivo

In [ ]:
tabela = pd.DataFrame([
    ("Dijkstra (heap)", "O((V+A) log V)", "1 origem → todos", "pesos ≥ 0", f"{np.mean(exp_d):,.0f}", "✅ base do projeto: pesos > 0; verificado vs NetworkX"),
    ("A* (haversine)",  "≈ Dijkstra, menos nós", "origem → destino", "heurística admissível/consistente", f"{np.mean(exp_a):,.0f}", f"✅ {100*(1-np.mean(exp_a)/np.mean(exp_d)):.0f}% menos nós, mesma distância ótima"),
    ("Dijkstra O(V²)",  "O(V²)", "1 origem → todos", "pesos ≥ 0", "—", "didático; inviável na cidade inteira (seção 4)"),
    ("Bellman-Ford",    "O(V·A)", "1 origem → todos", "aceita pesos negativos", "—", f"desnecessário (sem pesos negativos); ≈ {V*A:.1e} operações aqui"),
    ("Floyd-Warshall",  "O(V³)", "todos → todos", "aceita pesos negativos", "—", f"impraticável: ≈ {V**3:.1e} operações e matriz {V}×{V}"),
], columns=["Algoritmo", "Complexidade", "Resolve", "Hipótese", "Nós expandidos (média)", "Veredicto"])
display(tabela)

In [ ]:
pior = df_b.iloc[0]
print("RESUMO EXECUTIVO"); print("-" * 80)
print(f"• Base: rede viária real de Kraków (OpenStreetMap) — {V:,} nós, {A:,} arcos (dirigidos, com mão única); pesos em metros.")
print(f"• Método: Dijkstra e A* implementados do zero; 200 pares conferidos contra o NetworkX (0 divergências).")
print(f"• Desempenho: par origem–destino em {t_dij*1e3:.1f} ms (Dijkstra) e {t_ast*1e3:.1f} ms (A*, {100*(1-np.mean(exp_a)/np.mean(exp_d)):.0f}% menos nós).")
print(f"• Cenário: {'CASO REAL' if CASO_REAL else 'SIMULAÇÃO (pares amostrados)'} com {len(df)} viagens; baseline = menor nº de trechos.")
print(f"• Ganho: {KM_A:,.0f} → {KM_D:,.0f} km ({GANHO_PCT:.1f}% a menos); {MUDARAM} rotas mudaram; maior ganho individual {df.ganho_pct.max():.0f}%.")
print(f"• Tempo (hipótese {VEL_MEDIA_KMH:.0f} km/h): {MIN_A-MIN_D:,.0f} min a menos no conjunto.")
print(f"• Risco: bloquear o trecho mais crítico da rota-exemplo soma {pior.desvio_m:,.0f} m ({pior.desvio_pct:.1f}%).")
print(f"• Robustez: com erro de ±30% nos pesos, a distância extra média é {df_r.iloc[-1,2]:.2f}% (pior caso {df_r.iloc[-1,3]:.2f}%).")
print("• Limites: 'antes' é um critério de referência; velocidade média é hipótese; sem custo financeiro se CUSTO_POR_KM = None.")